In [1]:
!pip install -q -U transformers accelerate bitsandbytes peft qwen-vl-utils datasets "pillow!=12.0.0"


In [2]:
import os

# Reduces allocator fragmentation — genuinely helps on a tight-memory GPU.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import random
import time

import numpy as np
import pandas as pd
import PIL
from PIL import Image

try:
    from PIL._typing import _Ink  # noqa: F401
except ImportError:
    raise ImportError(
        f"Pillow {PIL.__version__} is missing PIL._typing._Ink (a known bug in "
        "pillow==12.0.0 specifically). Run: !pip install -q -U --force-reinstall "
        "\"pillow!=12.0.0\" then RESTART THE RUNTIME (Runtime > Restart session) "
        "before rerunning this cell — a plain pip install doesn't undo an already-"
        "imported broken module."
    )

import torch
from torch.utils.data import Dataset

from datasets import load_dataset

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from qwen_vl_utils import process_vision_info

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4


In [2]:
!pip install -q -U --force-reinstall "pillow!=12.0.0"

In [3]:
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

SEED = 123

# --- Memory controls -----------------------------------------------------
MAX_IMAGE_SIDE = 336
COMPUTE_DTYPE = torch.float16
IMAGE_MODE = "original"

# --- Data sizing -----------------------------------------------------------
EVAL_SIZE = 100
TRAIN_SIZE = 750
CHUNK_SIZE = 10

# --- Training --------------------------------------------------------------
LEARNING_RATE = 5e-5
GRADIENT_ACCUMULATION_STEPS = 4
MAX_GRAD_NORM = 1.0
LORA_R = 4
LORA_ALPHA = 8

# --- Generation ------------------------------------------------------------
MAX_NEW_TOKENS = 6

# ============================================================
# PERSISTENT STORAGE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/VCR_Qwen3VL"

CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
FINAL_ADAPTER = os.path.join(DRIVE_ROOT, "final_adapter")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Persistent checkpoint directory:")
print(CHECKPOINT_DIR)

print("\nDrive mounted:", os.path.exists("/content/drive"))
print("Checkpoint directory exists:", os.path.exists(CHECKPOINT_DIR))

# ============================================================
# SEEDS
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("\nModel:", MODEL_ID)
print(
    "Image side:", MAX_IMAGE_SIDE,
    "| compute dtype:", COMPUTE_DTYPE,
    "| image mode:", IMAGE_MODE
)
print(
    "Eval size:", EVAL_SIZE,
    "| train size:", TRAIN_SIZE,
    "| chunk size:", CHUNK_SIZE
)

Mounted at /content/drive
Persistent checkpoint directory:
/content/drive/MyDrive/VCR_Qwen3VL/checkpoints

Drive mounted: True
Checkpoint directory exists: True

Model: Qwen/Qwen3-VL-8B-Instruct
Image side: 336 | compute dtype: torch.float16 | image mode: original
Eval size: 100 | train size: 750 | chunk size: 10


In [4]:
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def gpu_status(label="GPU"):
    if not torch.cuda.is_available():
        print(label, ": CUDA unavailable")
        return

    free, total = torch.cuda.mem_get_info()
    used = total - free

    print(
        f"{label}: used={used / 1024**3:.2f} GB | "
        f"free={free / 1024**3:.2f} GB | "
        f"total={total / 1024**3:.2f} GB"
    )


cleanup()
gpu_status("Initial")


Initial: used=0.10 GB | free=14.46 GB | total=14.56 GB


In [5]:
questions_ds = load_dataset(
    "Rowan/vcr",
    "questions",
    split="train"
)

image_ds = load_dataset(
    "Rowan/vcr",
    "image_examples",
    split="train"
)

print("Questions:", len(questions_ds))
print("Images:", len(image_ds))


README.md:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

questions/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B / 76.3MB            

questions/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

questions/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B / 76.8MB            

...

Generating train split:   0%|          | 0/80418 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9929 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9557 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/47 [00:00<?, ?it/s]

Questions: 212923
Images: 80418


In [6]:
!du -sh /root/.cache/* 2>/dev/null | sort -hr | head -20

60G	/root/.cache/huggingface
142M	/root/.cache/pip
56M	/root/.cache/node-gyp
36K	/root/.cache/matplotlib
8.0K	/root/.cache/antigravity


In [7]:
!rm -rf /root/.cache/huggingface

In [8]:
image_index = {
    img_fn: i
    for i, img_fn in enumerate(image_ds["img_fn"])
}


class VCRDataset(Dataset):

    def __init__(self, questions, images, image_index):
        self.questions = questions
        self.images = images
        self.image_index = image_index

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):

        q = self.questions[idx]

        image_idx = self.image_index[q["img_fn"]]

        image = self.images[image_idx]["image"]

        return {
            "image": image,
            "question": q["question_text"],
            "answers": q["answer_choice_texts"],
            "answer_label": q["answer_label"],
            "rationales": q["rationale_choice_texts"],
            "rationale_label": q["rationale_label"],
            "boxes": q.get("boxes"),
            "img_fn": q["img_fn"]
        }


vcr_dataset = VCRDataset(
    questions_ds,
    image_ds,
    image_index
)

print("Dataset size:", len(vcr_dataset))


Dataset size: 212923


In [9]:
sample = vcr_dataset[0]

print("Question:", sample["question"])
print("Answers:", sample["answers"])
print("Correct answer:", sample["answer_label"])
print("Rationales:", sample["rationales"])
print("Correct rationale:", sample["rationale_label"])
print("Image size:", sample["image"].size)


Question: Does [person2] feel comfortable?
Answers: ['Yes because the person sitting next to her is smiling.', 'No she does not.', 'Yes, she is wearing something with thin straps.', 'Yes, she is cold.']
Correct answer: 1
Rationales: ['There is snow on the ground, and she is wearing a coat and hate.', 'She is standing with her arms crossed and looks disturbed.', 'She is sitting very rigidly and tensely on the edge of the bed. her posture is not relaxed and her face looks serious.', '[person2] is laying in bed but not sleeping. she looks sad and is curled into a ball.']
Correct rationale: 1
Image size: (1920, 804)


In [10]:
def answer_prompt(question, answers):

    return f"""You are solving Visual Commonsense Reasoning.

Your job is to select the ONE answer that is best supported by the image and question.

Pay attention to:
1. Which people/objects are referenced.
2. Their positions and interactions.
3. Actions and body language.
4. What is actually visible versus what is merely plausible.

Do not choose an answer just because it sounds reasonable — it must be supported by the image.

Question:
{question}

Answer choices:
0. {answers[0]}
1. {answers[1]}
2. {answers[2]}
3. {answers[3]}

Return only the number of the best answer: 0, 1, 2, or 3."""


def rationale_prompt(question, answer, rationales):

    return f"""You are solving Visual Commonsense Reasoning.

The answer has already been selected. Your job is to choose the ONE rationale that
actually explains why that answer is supported.

A good rationale must:
- agree with the image
- agree with the question
- agree with the selected answer
- avoid unsupported assumptions

Question:
{question}

Selected answer:
{answer}

Rationale choices:
0. {rationales[0]}
1. {rationales[1]}
2. {rationales[2]}
3. {rationales[3]}

Return only the number of the best rationale: 0, 1, 2, or 3."""


def extract_choice(output):
    import re
    match = re.search(r"\b([0-3])\b", output)
    return int(match.group(1)) if match else -1


Image preparation

In [11]:
_crop_warned = False


def _try_crop(image, boxes):

    global _crop_warned

    try:
        if not boxes:
            return image

        xs1, ys1, xs2, ys2 = [], [], [], []

        for box in boxes:
            if len(box) < 4:
                continue
            x1, y1, x2, y2 = box[0], box[1], box[2], box[3]
            xs1.append(x1); ys1.append(y1); xs2.append(x2); ys2.append(y2)

        if not xs1:
            return image

        # Union of all referenced boxes, plus a small padding margin.
        x1, y1, x2, y2 = min(xs1), min(ys1), max(xs2), max(ys2)

        w, h = image.size
        pad_x = 0.08 * (x2 - x1)
        pad_y = 0.08 * (y2 - y1)

        x1 = max(0, x1 - pad_x)
        y1 = max(0, y1 - pad_y)
        x2 = min(w, x2 + pad_x)
        y2 = min(h, y2 + pad_y)

        if x2 <= x1 or y2 <= y1:
            return image

        return image.crop((x1, y1, x2, y2))

    except Exception as e:
        if not _crop_warned:
            print(f"[crop] Falling back to original image (reason: {e}). This warning prints once.")
            _crop_warned = True
        return image


def prepare_image(sample):

    image = sample["image"].convert("RGB").copy()

    if IMAGE_MODE == "crop":
        image = _try_crop(image, sample.get("boxes"))

    image.thumbnail(
        (MAX_IMAGE_SIDE, MAX_IMAGE_SIDE),
        Image.Resampling.LANCZOS
    )

    return image


In [12]:
cleanup()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

gpu_status("After model load")
print("Model loaded.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

After model load: used=6.11 GB | free=8.45 GB | total=14.56 GB
Model loaded.


In [13]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
cleanup()
gpu_status("After LoRA")


trainable params: 3,833,856 || all params: 8,770,957,552 || trainable%: 0.0437
After LoRA: used=8.47 GB | free=6.09 GB | total=14.56 GB


In [14]:
def ask_qwen(sample, prompt):

    image = prepare_image(sample)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )

    new_tokens = generated[:, inputs["input_ids"].shape[1]:]

    output = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

    del inputs, generated, new_tokens
    cleanup()

    return output


In [15]:
model.eval()
model.config.use_cache = True

test_sample = vcr_dataset[0]

prompt = answer_prompt(test_sample["question"], test_sample["answers"])
output = ask_qwen(test_sample, prompt)

print("Model output:", repr(output))
print("Prediction:", extract_choice(output))
print("Correct:", test_sample["answer_label"])

gpu_status("After sanity generate")


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Model output: '1'
Prediction: 1
Correct: 1
After sanity generate: used=8.49 GB | free=6.08 GB | total=14.56 GB


Baseline evaluation

In [16]:
random.seed(SEED)

eval_indices = random.sample(range(len(vcr_dataset)), EVAL_SIZE)

print("Evaluation samples:", len(eval_indices))


Evaluation samples: 100


In [17]:
def run_answer_eval(tag):

    results = []

    for count, idx in enumerate(eval_indices):

        sample = vcr_dataset[idx]

        prompt = answer_prompt(sample["question"], sample["answers"])
        output = ask_qwen(sample, prompt)
        prediction = extract_choice(output)

        results.append({
            "index": idx,
            "true_answer": sample["answer_label"],
            "predicted_answer": prediction,
            "answer_correct": prediction == sample["answer_label"],
        })

        if (count + 1) % 10 == 0:
            print(f"[{tag}] {count + 1}/{EVAL_SIZE}")

    return pd.DataFrame(results)


def run_rationale_eval(tag):

    results = []

    for count, idx in enumerate(eval_indices):

        sample = vcr_dataset[idx]
        correct_answer = sample["answers"][sample["answer_label"]]

        prompt = rationale_prompt(sample["question"], correct_answer, sample["rationales"])
        output = ask_qwen(sample, prompt)
        prediction = extract_choice(output)

        results.append({
            "index": idx,
            "true_rationale": sample["rationale_label"],
            "predicted_rationale": prediction,
            "rationale_correct": prediction == sample["rationale_label"],
        })

        if (count + 1) % 10 == 0:
            print(f"[{tag}] {count + 1}/{EVAL_SIZE}")

    return pd.DataFrame(results)


def joint_accuracy(answer_df, rationale_df):
    both = answer_df["answer_correct"].values & rationale_df["rationale_correct"].values
    return both.mean()


In [18]:
baseline_answer_df = run_answer_eval("baseline-answer")
baseline_answer_accuracy = baseline_answer_df["answer_correct"].mean()
print("Baseline Q\u2192A:", f"{baseline_answer_accuracy:.2%}")

gpu_status("After baseline answer eval")


[baseline-answer] 10/100
[baseline-answer] 20/100
[baseline-answer] 30/100
[baseline-answer] 40/100
[baseline-answer] 50/100
[baseline-answer] 60/100
[baseline-answer] 70/100
[baseline-answer] 80/100
[baseline-answer] 90/100
[baseline-answer] 100/100
Baseline Q→A: 67.00%
After baseline answer eval: used=8.49 GB | free=6.08 GB | total=14.56 GB


In [ ]:
baseline_rationale_df = run_rationale_eval("baseline-rationale")
baseline_rationale_accuracy = baseline_rationale_df["rationale_correct"].mean()
print("Baseline QA\u2192R:", f"{baseline_rationale_accuracy:.2%}")


[baseline-rationale] 10/100
[baseline-rationale] 20/100
[baseline-rationale] 30/100
[baseline-rationale] 40/100
[baseline-rationale] 50/100
[baseline-rationale] 60/100
[baseline-rationale] 70/100
[baseline-rationale] 80/100
[baseline-rationale] 90/100
[baseline-rationale] 100/100
Baseline QA→R: 65.00%


In [ ]:
baseline_joint_accuracy = joint_accuracy(baseline_answer_df, baseline_rationale_df)

baseline_results = pd.DataFrame({
    "Metric": ["Q\u2192A (answer)", "QA\u2192R (rationale)", "Joint Q\u2192AR"],
    "Accuracy": [baseline_answer_accuracy, baseline_rationale_accuracy, baseline_joint_accuracy],
})

baseline_answer_df.to_csv(f"{RESULTS_DIR}/baseline_answer_results.csv", index=False)
baseline_rationale_df.to_csv(f"{RESULTS_DIR}/baseline_rationale_results.csv", index=False)
baseline_results.to_csv(f"{RESULTS_DIR}/baseline_summary.csv", index=False)

baseline_results


,Metric,Accuracy
0,Q→A (answer),0.67
1,QA→R (rationale),0.65
2,Joint Q→AR,0.40


Training setup

In [19]:
random.seed(SEED + 1)

train_pool = [i for i in range(len(vcr_dataset)) if i not in set(eval_indices)]
train_question_indices = random.sample(train_pool, TRAIN_SIZE)

print("Training questions:", len(train_question_indices))
print("Overlap with eval set:", len(set(train_question_indices) & set(eval_indices)))


Training questions: 750
Overlap with eval set: 0


In [20]:
def training_prompt_answer(sample):
    # Same phrasing as answer_prompt, minus the target — kept close to eval-time wording
    # so the model isn't learning one prompt style and being tested on another.
    return answer_prompt(sample["question"], sample["answers"])


def training_prompt_rationale(sample):
    correct_answer = sample["answers"][sample["answer_label"]]
    return rationale_prompt(sample["question"], correct_answer, sample["rationales"])


def build_training_instances(question_indices):
    """Turn each VCR question into an (idx, task) pair for both sub-tasks."""
    instances = []
    for idx in question_indices:
        instances.append((idx, "answer"))
        instances.append((idx, "rationale"))
    random.shuffle(instances)
    return instances


train_instances = build_training_instances(train_question_indices)

training_chunks = [
    train_instances[i:i + CHUNK_SIZE]
    for i in range(0, len(train_instances), CHUNK_SIZE)
]

print("Training instances (answer + rationale):", len(train_instances))
print("Chunks:", len(training_chunks))


Training instances (answer + rationale): 1500
Chunks: 150


In [21]:
def make_training_inputs(idx, task):

    sample = vcr_dataset[idx]
    image = prepare_image(sample)

    if task == "answer":
        prompt_text_raw = training_prompt_answer(sample)
        target_text = f" {sample['answer_label']}"
    elif task == "rationale":
        prompt_text_raw = training_prompt_rationale(sample)
        target_text = f" {sample['rationale_label']}"
    else:
        raise ValueError(f"Unknown task: {task}")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text_raw},
            ],
        }
    ]

    prompt_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    full_text = prompt_text + target_text

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[full_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    prompt_tokens = processor.tokenizer(
        prompt_text, add_special_tokens=False, return_tensors="pt"
    )
    prompt_length = prompt_tokens.input_ids.shape[1]

    labels = inputs["input_ids"].clone()
    labels[:, :prompt_length] = -100
    inputs["labels"] = labels

    return inputs


In [22]:
# ============================================================
# RESUME FROM GOOGLE DRIVE CHECKPOINT
# ============================================================

import os
import re

def get_latest_checkpoint():

    if not os.path.exists(CHECKPOINT_DIR):
        return None, 0

    checkpoints = []

    for name in os.listdir(CHECKPOINT_DIR):

        match = re.fullmatch(r"chunk_(\d+)", name)

        if match:
            number = int(match.group(1))
            path = os.path.join(CHECKPOINT_DIR, name)

            if os.path.isdir(path):
                checkpoints.append((number, path))

    if not checkpoints:
        return None, 0

    checkpoints.sort()

    return checkpoints[-1][1], checkpoints[-1][0]


latest_checkpoint, latest_chunk = get_latest_checkpoint()

if latest_checkpoint is None:

    print("No checkpoint found.")
    print("Starting from scratch.")

else:

    print("=" * 60)
    print("CHECKPOINT FOUND")
    print("=" * 60)

    print("Latest chunk:", latest_chunk)
    print("Path:", latest_checkpoint)

    print("\nLoading LoRA adapter...")

    from peft import PeftModel

    model = PeftModel.from_pretrained(
        model,
        latest_checkpoint,
        is_trainable=True,
    )

    model.train()

    print(
        f"\nSuccessfully resumed from chunk {latest_chunk}."
    )

    print(
        f"Next chunk will be {latest_chunk + 1}."
    )

    cleanup()
    gpu_status("After checkpoint restore")

No checkpoint found.
Starting from scratch.


### Sanity check: one forward + backward, with NaN and OOM guards

In [23]:
cleanup()
gpu_status("Before sanity test")

sanity_idx, sanity_task = train_instances[0]
inputs = make_training_inputs(sanity_idx, sanity_task)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

try:
    model.train()

    outputs = model(**inputs)
    loss = outputs.loss

    print("Task:", sanity_task, "| Loss:", float(loss))

    if not torch.isfinite(loss):
        raise RuntimeError("Loss is NaN/Inf on the very first example.")

    loss.backward()
    print("Forward + backward succeeded.")

except torch.cuda.OutOfMemoryError:
    print("OOM at MAX_IMAGE_SIDE =", MAX_IMAGE_SIDE)
    print("Fix: set MAX_IMAGE_SIDE = 280 in the config cell and rerun from the model-load cell.")
    raise

finally:
    model.zero_grad(set_to_none=True)
    del inputs, outputs, loss
    cleanup()
    gpu_status("After sanity test")


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Before sanity test: used=8.49 GB | free=6.08 GB | total=14.56 GB


/tmp/ipykernel_2534/907291913.py:14: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Task:", sanity_task, "| Loss:", float(loss))


Task: rationale | Loss: 4.075933456420898
Forward + backward succeeded.
After sanity test: used=8.49 GB | free=6.07 GB | total=14.56 GB


In [24]:
trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

optimizer.zero_grad(set_to_none=True)

print("Trainable parameters:", sum(p.numel() for p in trainable_params))


Trainable parameters: 3833856


In [25]:
def train_chunk(chunk_instances, chunk_number):

    model.train()

    total_loss = 0.0
    n_successful = 0
    consecutive_ooms = 0

    optimizer.zero_grad(set_to_none=True)

    for step, (idx, task) in enumerate(chunk_instances):

        try:
            inputs = make_training_inputs(idx, task)
            inputs = {k: v.to(model.device, non_blocking=True) for k, v in inputs.items()}

            outputs = model(**inputs)
            raw_loss = outputs.loss

            if not torch.isfinite(raw_loss):
                print(f"NaN/Inf loss at chunk {chunk_number}, step {step + 1} (idx={idx}, task={task}).")
                del inputs, outputs, raw_loss
                cleanup()
                return None, False, n_successful

            total_loss += raw_loss.item()
            n_successful += 1
            consecutive_ooms = 0

            loss = raw_loss / GRADIENT_ACCUMULATION_STEPS
            loss.backward()

            boundary = (
                (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
                or step + 1 == len(chunk_instances)
            )

            if boundary:
                grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)

                if not torch.isfinite(grad_norm):
                    print(f"Non-finite gradient norm at chunk {chunk_number}, step {step + 1}.")
                    del inputs, outputs, raw_loss, loss
                    cleanup()
                    return None, False, n_successful

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            del inputs, outputs, raw_loss, loss
            cleanup()

        except torch.cuda.OutOfMemoryError:
            consecutive_ooms += 1
            optimizer.zero_grad(set_to_none=True)
            # Not all of these are guaranteed to be bound depending on where the OOM hit,
            # so clear what we can and lean on cleanup() below for the rest.
            inputs = outputs = raw_loss = loss = None
            cleanup()

            print(f"OOM on chunk {chunk_number}, step {step + 1} (idx={idx}, task={task}) — skipping this example.")

            if consecutive_ooms >= 3:
                print("Three OOMs in a row. Lower MAX_IMAGE_SIDE (e.g. to 280) and restart from the model-load cell.")
                return None, False, n_successful

            continue

        if (step + 1) % 5 == 0:
            print(f"Chunk {chunk_number} | {step + 1}/{len(chunk_instances)}")
            gpu_status("GPU")

    if n_successful == 0:
        return None, False, n_successful

    return total_loss / n_successful, True, n_successful


Train


In [27]:
# ============================================================
# RESUMABLE + GPU-DIE-SAFE TRAINING LOOP
# ============================================================

import os
import re
import json
import time

training_losses = []
last_good_checkpoint = None
training_stable = True


def get_completed_chunks():

    if not os.path.exists(CHECKPOINT_DIR):
        return []

    completed = []

    for name in os.listdir(CHECKPOINT_DIR):

        match = re.fullmatch(r"chunk_(\d+)", name)

        if match:
            chunk_number = int(match.group(1))
            path = os.path.join(CHECKPOINT_DIR, name)

            if os.path.isdir(path):
                completed.append(chunk_number)

    return sorted(completed)


completed_chunks = get_completed_chunks()

if completed_chunks:

    last_completed_chunk = max(completed_chunks)

    print("=" * 60)
    print("RESUMING TRAINING")
    print("=" * 60)

    print("Completed chunks found:", len(completed_chunks))
    print("Latest completed chunk:", last_completed_chunk)

    START_CHUNK = last_completed_chunk + 1

else:

    print("=" * 60)
    print("STARTING TRAINING FROM CHUNK 1")
    print("=" * 60)

    START_CHUNK = 1


print("Starting from chunk:", START_CHUNK)
print("Total chunks:", len(training_chunks))


# ============================================================
# TRAIN
# ============================================================

for chunk_number in range(START_CHUNK, len(training_chunks) + 1):

    chunk_instances = training_chunks[chunk_number - 1]

    print("\n" + "=" * 60)
    print(f"CHUNK {chunk_number}/{len(training_chunks)}")
    print("=" * 60)

    avg_loss, chunk_ok, n_successful = train_chunk(
        chunk_instances,
        chunk_number
    )

    # --------------------------------------------------------
    # CHUNK FAILED
    # --------------------------------------------------------

    if not chunk_ok:

        training_stable = False

        print(
            f"\nChunk {chunk_number} failed."
        )

        print(
            "Latest persistent checkpoint:",
            last_good_checkpoint
        )

        break

    # --------------------------------------------------------
    # SAVE CHECKPOINT DIRECTLY TO GOOGLE DRIVE
    # --------------------------------------------------------

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"chunk_{chunk_number}"
    )

    model.save_pretrained(checkpoint_path)
    processor.save_pretrained(checkpoint_path)

    # --------------------------------------------------------
    # VERIFY CHECKPOINT
    # --------------------------------------------------------

    adapter_file = os.path.join(
        checkpoint_path,
        "adapter_model.safetensors"
    )

    adapter_bin = os.path.join(
        checkpoint_path,
        "adapter_model.bin"
    )

    checkpoint_verified = (
        os.path.exists(adapter_file)
        or os.path.exists(adapter_bin)
    )

    if not checkpoint_verified:

        raise RuntimeError(
            f"CHECKPOINT VERIFICATION FAILED for chunk {chunk_number}.\n"
            f"Path: {checkpoint_path}\n"
            "Training stopped to prevent losing progress."
        )

    # --------------------------------------------------------
    # WRITE PROGRESS FILE
    # --------------------------------------------------------

    progress = {
        "last_completed_chunk": chunk_number,
        "total_chunks": len(training_chunks),
        "avg_loss": float(avg_loss),
        "successful_examples": n_successful,
        "checkpoint_path": checkpoint_path,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    progress_path = os.path.join(
        DRIVE_ROOT,
        "training_progress.json"
    )

    with open(progress_path, "w") as f:
        json.dump(progress, f, indent=2)

    # --------------------------------------------------------
    # BOOKKEEPING
    # --------------------------------------------------------

    training_losses.append(avg_loss)
    last_good_checkpoint = checkpoint_path

    print(
        f"Average loss: {avg_loss:.5f} "
        f"({n_successful}/{len(chunk_instances)} examples used)"
    )

    print("Checkpoint saved:", checkpoint_path)
    print("Checkpoint VERIFIED:", checkpoint_verified)

    cleanup()

    gpu_status(
        f"After chunk {chunk_number}"
    )

    print(
        f"Progress: {chunk_number}/{len(training_chunks)}"
    )

    time.sleep(2)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 60)
print("TRAINING STATUS")
print("=" * 60)

print("Training stable:", training_stable)
print("Last good checkpoint:", last_good_checkpoint)

completed_chunks = get_completed_chunks()

print(
    "Persistent completed chunks:",
    len(completed_chunks)
)

if completed_chunks:
    print(
        "Latest persistent chunk:",
        max(completed_chunks)
    )

RESUMING TRAINING
Completed chunks found: 150
Latest completed chunk: 150
Starting from chunk: 151
Total chunks: 150

TRAINING STATUS
Training stable: True
Last good checkpoint: None
Persistent completed chunks: 150
Latest persistent chunk: 150


In [28]:
# ============================================================
# SAVE FINAL ADAPTER TO GOOGLE DRIVE
# ============================================================

completed_chunks = get_completed_chunks()

if not completed_chunks:
    raise RuntimeError(
        "No persistent checkpoint exists. "
        "Training did not complete even one chunk."
    )

latest_chunk = max(completed_chunks)

latest_checkpoint = os.path.join(
    CHECKPOINT_DIR,
    f"chunk_{latest_chunk}"
)

print("Latest checkpoint:", latest_checkpoint)

if latest_chunk < len(training_chunks):

    print(
        f"Training stopped at chunk {latest_chunk}/{len(training_chunks)}."
    )

    print(
        "Keeping the latest checkpoint safely in Google Drive."
    )

else:

    print("ALL TRAINING CHUNKS COMPLETED.")


# Save final adapter from current model state
model.save_pretrained(FINAL_ADAPTER)
processor.save_pretrained(FINAL_ADAPTER)

print("\nFINAL ADAPTER SAVED:")
print(FINAL_ADAPTER)

Latest checkpoint: /content/drive/MyDrive/VCR_Qwen3VL/checkpoints/chunk_150
ALL TRAINING CHUNKS COMPLETED.

FINAL ADAPTER SAVED:
/content/drive/MyDrive/VCR_Qwen3VL/final_adapter


In [29]:
model.eval()
model.config.use_cache = True
cleanup()
gpu_status("QLoRA model ready")


QLoRA model ready: used=8.53 GB | free=6.04 GB | total=14.56 GB


In [30]:
qlora_answer_df = run_answer_eval("qlora-answer")
qlora_answer_accuracy = qlora_answer_df["answer_correct"].mean()
print("QLoRA Q\u2192A:", f"{qlora_answer_accuracy:.2%}")


[qlora-answer] 10/100
[qlora-answer] 20/100
[qlora-answer] 30/100
[qlora-answer] 40/100
[qlora-answer] 50/100
[qlora-answer] 60/100
[qlora-answer] 70/100
[qlora-answer] 80/100
[qlora-answer] 90/100
[qlora-answer] 100/100
QLoRA Q→A: 71.00%


In [31]:
qlora_rationale_df = run_rationale_eval("qlora-rationale")
qlora_rationale_accuracy = qlora_rationale_df["rationale_correct"].mean()
print("QLoRA QA\u2192R:", f"{qlora_rationale_accuracy:.2%}")


[qlora-rationale] 10/100
[qlora-rationale] 20/100
[qlora-rationale] 30/100
[qlora-rationale] 40/100
[qlora-rationale] 50/100
[qlora-rationale] 60/100
[qlora-rationale] 70/100
[qlora-rationale] 80/100
[qlora-rationale] 90/100
[qlora-rationale] 100/100
QLoRA QA→R: 72.00%


In [33]:
qlora_joint_accuracy = joint_accuracy(qlora_answer_df, qlora_rationale_df)



In [34]:
print(qlora_joint_accuracy)

0.48


In [37]:
# SAFE VARIABLE INSPECTION
items = list(globals().items())

for name, value in items:
    if any(x in name.lower() for x in ["train", "chunk", "data", "instance"]):
        try:
            print(f"{name:35} | {type(value).__name__:20} | len={len(value)}")
        except Exception:
            print(f"{name:35} | {type(value).__name__:20}")

Dataset                             | type                
load_dataset                        | function            
prepare_model_for_kbit_training     | function            
TRAIN_SIZE                          | int                 
CHUNK_SIZE                          | int                 
VCRDataset                          | type                
vcr_dataset                         | VCRDataset           | len=212923
train_pool                          | list                 | len=212823
train_question_indices              | list                 | len=750
training_prompt_answer              | function            
training_prompt_rationale           | function            
build_training_instances            | function            
train_instances                     | list                 | len=1500
training_chunks                     | list                 | len=150
make_training_inputs                | function            
latest_chunk                        | int                 

In [38]:
# ============================================================
# SAVE CURRENT 750-QUESTION MODEL SAFELY
# ============================================================

from google.colab import drive
import os

# Mount Drive if not already mounted
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/VCR_Qwen3VL"
FINAL_750_ADAPTER = os.path.join(DRIVE_ROOT, "final_adapter_750")

os.makedirs(FINAL_750_ADAPTER, exist_ok=True)

print("Saving current model...")
print("Completed chunks:", len(completed_chunks))
print("Latest chunk:", max(completed_chunks))

model.save_pretrained(FINAL_750_ADAPTER)
processor.save_pretrained(FINAL_750_ADAPTER)

print("\n✅ SAVED!")
print(FINAL_750_ADAPTER)

Saving current model...
Completed chunks: 150
Latest chunk: 150

✅ SAVED!
/content/drive/MyDrive/VCR_Qwen3VL/final_adapter_750


In [40]:
#

In [41]:
# ============================================================
# CONTINUE TRAINING: 750 → 1000 QUESTIONS
# ============================================================

import os
import random
import json
import time

EXTRA_QUESTIONS = 250
CHUNK_SIZE = 10

# ------------------------------------------------------------
# 1. SELECT 250 NEW QUESTIONS
# ------------------------------------------------------------

# Current 750 questions are already in train_question_indices.
# Pick 250 from the remaining training pool, avoiding overlap.

used_questions = set(train_question_indices)

remaining_pool = [
    idx for idx in train_pool
    if idx not in used_questions
]

print("Already trained questions :", len(used_questions))
print("Remaining available      :", len(remaining_pool))

assert len(remaining_pool) >= EXTRA_QUESTIONS

# Deterministic selection
rng = random.Random(SEED + 2)

extra_question_indices = rng.sample(
    remaining_pool,
    EXTRA_QUESTIONS
)

print("New questions selected    :", len(extra_question_indices))

# ------------------------------------------------------------
# 2. BUILD ANSWER + RATIONALE INSTANCES
# ------------------------------------------------------------

extra_instances = build_training_instances(
    extra_question_indices
)

print("New training instances    :", len(extra_instances))

# 250 questions × 2 tasks = 500 instances
assert len(extra_instances) == 500

# ------------------------------------------------------------
# 3. SPLIT INTO 50 CHUNKS
# ------------------------------------------------------------

extra_chunks = [
    extra_instances[i:i + CHUNK_SIZE]
    for i in range(
        0,
        len(extra_instances),
        CHUNK_SIZE
    )
]

print("New chunks                :", len(extra_chunks))

assert len(extra_chunks) == 50

# ------------------------------------------------------------
# 4. PERSISTENT DRIVE LOCATION
# ------------------------------------------------------------

DRIVE_ROOT = "/content/drive/MyDrive/VCR_Qwen3VL"

CONTINUE_DIR = os.path.join(
    DRIVE_ROOT,
    "continued_750_to_1000"
)

os.makedirs(CONTINUE_DIR, exist_ok=True)

print("\nCheckpoint directory:")
print(CONTINUE_DIR)

# ------------------------------------------------------------
# 5. DETECT ALREADY COMPLETED CONTINUATION CHUNKS
# ------------------------------------------------------------

completed_extra = []

for name in os.listdir(CONTINUE_DIR):

    if name.startswith("chunk_"):

        try:
            n = int(name.split("_")[1])

            path = os.path.join(
                CONTINUE_DIR,
                name
            )

            if os.path.isdir(path):
                completed_extra.append(n)

        except:
            pass

completed_extra = sorted(completed_extra)

print("\nAlready completed continuation chunks:")
print(completed_extra)

# ------------------------------------------------------------
# 6. CONTINUE TRAINING
# ------------------------------------------------------------

extra_losses = []

if completed_extra:
    START_EXTRA = max(completed_extra) - 150 + 1
else:
    START_EXTRA = 1

print("\nStarting additional chunk:", START_EXTRA)
print("Total additional chunks :", len(extra_chunks))

for extra_chunk_number in range(
    START_EXTRA,
    len(extra_chunks) + 1
):

    global_chunk = 150 + extra_chunk_number

    # If checkpoint already exists, skip it
    checkpoint_path = os.path.join(
        CONTINUE_DIR,
        f"chunk_{global_chunk}"
    )

    if os.path.exists(checkpoint_path):
        print(
            f"⏭️ Chunk {global_chunk} already saved. Skipping."
        )
        continue

    chunk_instances = extra_chunks[
        extra_chunk_number - 1
    ]

    print("\n" + "=" * 60)
    print(
        f"CHUNK {global_chunk}/200"
    )
    print(
        f"Additional: {extra_chunk_number}/50"
    )
    print("=" * 60)

    avg_loss, chunk_ok, n_successful = train_chunk(
        chunk_instances,
        global_chunk
    )

    # --------------------------------------------------------
    # FAILURE
    # --------------------------------------------------------

    if not chunk_ok:

        print(
            f"\n❌ Chunk {global_chunk} failed."
        )

        print(
            "Stopping safely."
        )

        break

    # --------------------------------------------------------
    # SAVE TO GOOGLE DRIVE
    # --------------------------------------------------------

    os.makedirs(
        checkpoint_path,
        exist_ok=True
    )

    model.save_pretrained(
        checkpoint_path
    )

    processor.save_pretrained(
        checkpoint_path
    )

    # --------------------------------------------------------
    # VERIFY CHECKPOINT
    # --------------------------------------------------------

    adapter_file = os.path.join(
        checkpoint_path,
        "adapter_model.safetensors"
    )

    adapter_bin = os.path.join(
        checkpoint_path,
        "adapter_model.bin"
    )

    verified = (
        os.path.exists(adapter_file)
        or os.path.exists(adapter_bin)
    )

    if not verified:

        raise RuntimeError(
            f"CHECKPOINT VERIFICATION FAILED:\n"
            f"{checkpoint_path}"
        )

    extra_losses.append(avg_loss)

    print(
        f"Average loss: {avg_loss:.5f}"
    )

    print(
        f"Examples: {n_successful}/{len(chunk_instances)}"
    )

    print(
        f"💾 Saved + verified:"
    )

    print(
        checkpoint_path
    )

    cleanup()

    gpu_status(
        f"After chunk {global_chunk}"
    )

    # --------------------------------------------------------
    # PROGRESS FILE
    # --------------------------------------------------------

    progress = {
        "base_questions": 750,
        "extra_questions": 250,
        "total_questions": 750 + extra_chunk_number * 5,
        "last_completed_chunk": global_chunk,
        "additional_chunks_completed": extra_chunk_number,
        "total_additional_chunks": 50,
        "avg_loss": float(avg_loss),
        "timestamp": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

    with open(
        os.path.join(
            CONTINUE_DIR,
            "progress.json"
        ),
        "w"
    ) as f:

        json.dump(
            progress,
            f,
            indent=2
        )

    time.sleep(2)


# ============================================================
# FINAL STATUS
# ============================================================

completed_extra = []

for name in os.listdir(CONTINUE_DIR):

    if name.startswith("chunk_"):

        try:
            n = int(name.split("_")[1])

            if os.path.isdir(
                os.path.join(CONTINUE_DIR, name)
            ):
                completed_extra.append(n)

        except:
            pass

completed_extra = sorted(completed_extra)

print("\n" + "=" * 60)
print("CONTINUATION STATUS")
print("=" * 60)

print(
    "Completed chunks:",
    len(completed_extra),
    "/ 50"
)

if completed_extra:
    print(
        "Latest chunk:",
        max(completed_extra)
    )

if len(completed_extra) == 50:

    print("\n🎯 1000-QUESTION TRAINING COMPLETE!")

    FINAL_1000 = os.path.join(
        DRIVE_ROOT,
        "final_adapter_1000"
    )

    os.makedirs(
        FINAL_1000,
        exist_ok=True
    )

    model.save_pretrained(
        FINAL_1000
    )

    processor.save_pretrained(
        FINAL_1000
    )

    print(
        "Final adapter:",
        FINAL_1000
    )

else:

    print(
        "\n⚠️ Training incomplete."
    )

    print(
        "Latest checkpoint is safely on Google Drive."
    )

Already trained questions : 750
Remaining available      : 212073
New questions selected    : 250
New training instances    : 500
New chunks                : 50

Checkpoint directory:
/content/drive/MyDrive/VCR_Qwen3VL/continued_750_to_1000

Already completed continuation chunks:
[]

Starting additional chunk: 1
Total additional chunks : 50

CHUNK 151/200
Additional: 1/50


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Chunk 151 | 5/10
GPU: used=8.54 GB | free=6.03 GB | total=14.56 GB
Chunk 151 | 10/10
GPU: used=8.53 GB | free=6.04 GB | total=14.56 GB
Average loss: 1.06632
Examples: 10/10
💾 Saved + verified:
/content/drive/MyDrive/VCR_Qwen3VL/continued_750_to_1000/chunk_151
After chunk 151: used=8.53 GB | free=6.04 GB | total=14.56 GB

CHUNK 152/200
Additional: 2/50
Chunk 152 | 5/10
GPU: used=8.54 GB | free=6.03 GB | total=14.56 GB
Chunk 152 | 10/10
GPU: used=8.53 GB | free=6.04 GB | total=14.56 GB
Average loss: 1.14752
Examples: 10/10
💾 Saved + verified:
/content/drive/MyDrive/VCR_Qwen3VL/continued_750_to_1000/chunk_152
After chunk 152: used=8.53 GB | free=6.04 GB | total=14.56 GB

CHUNK 153/200
Additional: 3/50
Chunk 153 | 5/10
GPU: used=8.54 GB | free=6.03 GB | total=14.56 GB
Chunk 153 | 10/10
GPU: used=8.53 GB | free=6.04 GB | total=14.56 GB
Average loss: 0.93196
Examples: 10/10
💾 Saved + verified:
/content/drive/MyDrive/VCR_Qwen3VL/continued_750_to_1000/chunk_153
After chunk 153: used=8.53 GB | 

In [43]:
qlora_answer_df = run_answer_eval("qlora-answer")
qlora_answer_accuracy = qlora_answer_df["answer_correct"].mean()
print("QLoRA Q\u2192A:", f"{qlora_answer_accuracy:.2%}")

[qlora-answer] 10/100
[qlora-answer] 20/100
[qlora-answer] 30/100
[qlora-answer] 40/100
[qlora-answer] 50/100
[qlora-answer] 60/100
[qlora-answer] 70/100
[qlora-answer] 80/100
[qlora-answer] 90/100
[qlora-answer] 100/100
QLoRA Q→A: 74.00%


In [45]:
qlora_rationale_df = run_rationale_eval("qlora-rationale")
qlora_rationale_accuracy = qlora_rationale_df["rationale_correct"].mean()
print("QLoRA QA\u2192R:", f"{qlora_rationale_accuracy:.2%}")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[qlora-rationale] 10/100
[qlora-rationale] 20/100
[qlora-rationale] 30/100
[qlora-rationale] 40/100
[qlora-rationale] 50/100
[qlora-rationale] 60/100
[qlora-rationale] 70/100
[qlora-rationale] 80/100
[qlora-rationale] 90/100
[qlora-rationale] 100/100
QLoRA QA→R: 77.00%


In [46]:
qlora_joint_accuracy = joint_accuracy(qlora_answer_df, qlora_rationale_df)
print(qlora_joint_accuracy)

0.54


In [47]:
# ============================================================
# 1000-SAMPLE EVALUATION
# Same 1000 samples used for Answer + Rationale + Joint
# ============================================================

EVAL_SIZE = 1000

# Fixed seed so this exact evaluation set can be reused later
EVAL_SEED = 2026

eval_rng = random.Random(EVAL_SEED)

eval_indices = eval_rng.sample(
    range(len(vcr_dataset)),
    EVAL_SIZE
)

print("Evaluation samples:", len(eval_indices))
print("Evaluation seed:", EVAL_SEED)

# Save the exact indices to Drive
eval_indices_path = os.path.join(
    RESULTS_DIR,
    "eval_indices_1000.json"
)

with open(eval_indices_path, "w") as f:
    json.dump(eval_indices, f)

print("Saved indices:", eval_indices_path)

Evaluation samples: 1000
Evaluation seed: 2026
Saved indices: /content/drive/MyDrive/VCR_Qwen3VL/results/eval_indices_1000.json


In [48]:
# ============================================================
# 1000-SAMPLE ANSWER EVALUATION
# ============================================================

qlora_answer_1000_df = run_answer_eval("qlora-answer-1000")

qlora_answer_1000_accuracy = (
    qlora_answer_1000_df["answer_correct"].mean()
)

print(
    "\nQLoRA Q→A:",
    f"{qlora_answer_1000_accuracy:.2%}"
)

[qlora-answer-1000] 10/1000
[qlora-answer-1000] 20/1000
[qlora-answer-1000] 30/1000
[qlora-answer-1000] 40/1000
[qlora-answer-1000] 50/1000
[qlora-answer-1000] 60/1000
[qlora-answer-1000] 70/1000
[qlora-answer-1000] 80/1000
[qlora-answer-1000] 90/1000
[qlora-answer-1000] 100/1000
[qlora-answer-1000] 110/1000
[qlora-answer-1000] 120/1000
[qlora-answer-1000] 130/1000
[qlora-answer-1000] 140/1000
[qlora-answer-1000] 150/1000
[qlora-answer-1000] 160/1000
[qlora-answer-1000] 170/1000
[qlora-answer-1000] 180/1000
[qlora-answer-1000] 190/1000
[qlora-answer-1000] 200/1000
[qlora-answer-1000] 210/1000
[qlora-answer-1000] 220/1000
[qlora-answer-1000] 230/1000
[qlora-answer-1000] 240/1000
[qlora-answer-1000] 250/1000
[qlora-answer-1000] 260/1000
[qlora-answer-1000] 270/1000
[qlora-answer-1000] 280/1000
[qlora-answer-1000] 290/1000
[qlora-answer-1000] 300/1000
[qlora-answer-1000] 310/1000
[qlora-answer-1000] 320/1000
[qlora-answer-1000] 330/1000
[qlora-answer-1000] 340/1000
[qlora-answer-1000] 350

In [49]:
# ============================================================
# 1000-SAMPLE RATIONALE EVALUATION
# ============================================================

qlora_rationale_1000_df = run_rationale_eval(
    "qlora-rationale-1000"
)

qlora_rationale_1000_accuracy = (
    qlora_rationale_1000_df["rationale_correct"].mean()
)

print(
    "\nQLoRA QA→R:",
    f"{qlora_rationale_1000_accuracy:.2%}"
)

[qlora-rationale-1000] 10/1000
[qlora-rationale-1000] 20/1000
[qlora-rationale-1000] 30/1000
[qlora-rationale-1000] 40/1000
[qlora-rationale-1000] 50/1000
[qlora-rationale-1000] 60/1000
[qlora-rationale-1000] 70/1000
[qlora-rationale-1000] 80/1000
[qlora-rationale-1000] 90/1000
[qlora-rationale-1000] 100/1000
[qlora-rationale-1000] 110/1000
[qlora-rationale-1000] 120/1000
[qlora-rationale-1000] 130/1000
[qlora-rationale-1000] 140/1000
[qlora-rationale-1000] 150/1000
[qlora-rationale-1000] 160/1000
[qlora-rationale-1000] 170/1000
[qlora-rationale-1000] 180/1000
[qlora-rationale-1000] 190/1000
[qlora-rationale-1000] 200/1000
[qlora-rationale-1000] 210/1000
[qlora-rationale-1000] 220/1000
[qlora-rationale-1000] 230/1000
[qlora-rationale-1000] 240/1000
[qlora-rationale-1000] 250/1000
[qlora-rationale-1000] 260/1000
[qlora-rationale-1000] 270/1000
[qlora-rationale-1000] 280/1000
[qlora-rationale-1000] 290/1000
[qlora-rationale-1000] 300/1000
[qlora-rationale-1000] 310/1000
[qlora-rationale-

In [50]:
# ============================================================
# 1000-SAMPLE JOINT ACCURACY
# ============================================================

qlora_joint_1000_accuracy = joint_accuracy(
    qlora_answer_1000_df,
    qlora_rationale_1000_df
)

print(
    "QLoRA Joint Q→AR:",
    f"{qlora_joint_1000_accuracy:.2%}"
)

QLoRA Joint Q→AR: 54.00%
